# Parte 2 — Dataset: preparação, validação e divisão

## Dataset_EscutIA

Nesta aula vamos preparar o dataset que poderá apoiar a capacidade da EscutIA de identificar sentimentos em textos em português. A preparação inicial é gerada por `scripts/preparar_dataset.py` diretamente em `dados/`; este notebook explica cada decisão antes do gate final.

> A EscutIA é um agente para escuta, reflexão e bem-estar emocional. O modelo desta etapa é apenas uma capacidade especializada: ele não faz diagnóstico, não substitui psicólogos e não oferece tratamento psicológico.

Vamos trabalhar em 11 passos. Em cada passo, leia a explicação, execute somente as células daquele passo e confira o resultado antes de continuar.

## Antes de começar

Abra o terminal na pasta `EscutIA/dataset` e instale as dependências antes de abrir este notebook:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
```

Coloque o dataset local em `dados/dataset_local.csv`. Ao executar as células na ordem, o notebook baixa e unifica a fonte automaticamente usando a revisão fixada. O aluno acompanha o resultado na própria célula e os artefatos ficam diretamente em `dados/`.

In [16]:
# A primeira célula executável prepara o ambiente da aula; não é necessário abrir o terminal.
EXECUTAR_DOWNLOAD = True

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import re
import shutil
import subprocess
import sys
import unicodedata

import pandas as pd
from IPython.display import display
from jsonschema import Draft202012Validator
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'dados').exists() and (BASE_DIR / 'EscutIA' / 'dataset' / 'dados').exists():
    BASE_DIR = BASE_DIR / 'EscutIA' / 'dataset'

LOCAL_DATASET = BASE_DIR / 'dados' / 'dataset_local.csv'
DOWNLOAD_SCRIPT = BASE_DIR / 'scripts' / 'baixar_dataset.py'
DATASET_UNIFICADO = BASE_DIR / 'dados' / 'dataset.csv'
TRABALHO = BASE_DIR / 'dados' / 'trabalho'
PREPARADOS = BASE_DIR / 'dados' / 'preparados'
RELATORIOS = BASE_DIR / 'dados' / 'relatorios'
ROTULOS = {'negativo', 'neutro', 'positivo'}
# Cada execução completa começa limpa, sem apagar dataset_local.csv.
LIMPAR_RESULTADOS_ANTERIORES = True

PASTAS_GERADAS = [TRABALHO, PREPARADOS, RELATORIOS]
for pasta in PASTAS_GERADAS:
    pasta.mkdir(parents=True, exist_ok=True)
    if LIMPAR_RESULTADOS_ANTERIORES:
        for item in pasta.iterdir():
            if item.name == '.gitkeep':
                continue
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()

print('Resultados anteriores limpos: dados/trabalho, dados/preparados e dados/relatorios') if LIMPAR_RESULTADOS_ANTERIORES else None
print(f'Pasta da aula: {BASE_DIR}')


Resultados anteriores limpos: dados/trabalho, dados/preparados e dados/relatorios
Pasta da aula: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\dataset


In [17]:
def sha256_arquivo(arquivo):
    digest = hashlib.sha256()
    with arquivo.open('rb') as stream:
        for bloco in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(bloco)
    return digest.hexdigest()

def normalizar_texto(valor):
    valor = unicodedata.normalize('NFKC', str(valor or ''))
    return re.sub(r'\s+', ' ', valor).strip()

def chave_comparacao(valor):
    valor = normalizar_texto(valor).casefold()
    return re.sub(r'[^\w\s]', '', valor, flags=re.UNICODE)

def salvar_json_sem_sobrescrever(arquivo, conteudo):
    destino = Path(arquivo)
    destino.write_text(json.dumps(conteudo, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return destino

def salvar_jsonl_sem_sobrescrever(arquivo, registros):
    destino = Path(arquivo)
    destino.write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in registros), encoding='utf-8')
    return destino

def salvar_json_sem_sobrescrever_na_pasta_relatorios(nome, conteudo):
    return salvar_json_sem_sobrescrever(RELATORIOS / nome, conteudo)

## Passo 1 — Baixar e unificar o dataset

Vamos usar o script `scripts/baixar_dataset.py`. Ele baixa os splits `train`, `validation` e `test` do dataset de sentimentos em português, converte os rótulos para `negativo`, `neutro` e `positivo`, une os dados locais e remove textos duplicados.

Esta é a etapa de baixar e unificar. A célula executa automaticamente o script interno de download, combina a fonte local com os splits pinados e mostra quantidade, campos, hash e distribuição dos rótulos.

In [18]:
if not LOCAL_DATASET.exists():
    raise FileNotFoundError(f'Coloque o dataset local em: {LOCAL_DATASET}')
if not DOWNLOAD_SCRIPT.exists():
    raise FileNotFoundError(f'Script de download não encontrado: {DOWNLOAD_SCRIPT}')
if EXECUTAR_DOWNLOAD:
    subprocess.run([sys.executable, str(DOWNLOAD_SCRIPT)], cwd=BASE_DIR, check=True)
elif not DATASET_UNIFICADO.exists():
    raise FileNotFoundError('O dataset unificado ainda não existe. Execute novamente as células desde o início para refazer a etapa de download.')
else:
    print(f'Usando dataset unificado existente: {DATASET_UNIFICADO}')

df_original = pd.read_csv(DATASET_UNIFICADO)
print(f'Registros: {len(df_original)}')
print(f'Campos: {list(df_original.columns)}')
print(f'SHA-256 da fonte: {sha256_arquivo(DATASET_UNIFICADO)}')
display(df_original.head())
display(df_original.isna().sum().rename('valores_vazios').to_frame())
display(df_original['rotulo'].value_counts(dropna=False).rename('quantidade').to_frame())

Registros: 3063
Campos: ['id', 'texto', 'rotulo']
SHA-256 da fonte: 74c0c3b15ad86a6f54afb8fc9837d4e717f7894a4a9f85c5d58527705b70f81c


,id,texto,rotulo
0,1,Acordei animado e com vontade de começar o dia.,positivo
1,2,Foi muito bom conversar com meus amigos ontem.,positivo
2,3,Estou orgulhoso do resultado que consegui no t...,positivo
3,4,Hoje encontrei uma solução para um problema di...,positivo
4,5,Sinto esperança de que as coisas vão melhorar.,positivo


,valores_vazios
id,0
texto,0
rotulo,0


,quantidade
rotulo,
positivo,1021
neutro,1021
negativo,1021


In [19]:
relatorio_01 = {
    'passo': 1,
    'arquivo': str(DATASET_UNIFICADO),
    'dataset_local': str(LOCAL_DATASET),
    'script_unificacao': str(DOWNLOAD_SCRIPT),
    'sha256_dataset_local': sha256_arquivo(LOCAL_DATASET),
    'sha256': sha256_arquivo(DATASET_UNIFICADO),
    'registros': len(df_original),
    'campos': list(df_original.columns),
    'vazios': df_original.isna().sum().to_dict(),
    'distribuicao_rotulos': df_original['rotulo'].value_counts(dropna=False).to_dict(),
}
salvar_json_sem_sobrescrever_na_pasta_relatorios('01_inspecao.json', relatorio_01)

WindowsPath('c:/Users/mdbaa/development/alura/alura-llama-factory/EscutIA/dataset/dados/relatorios/01_inspecao.json')

## Passo 2 — Definir o schema dos exemplos

Cada registro precisa ter um identificador, um texto e um rótulo permitido. Se esta célula falhar, pare, corrija a fonte ou a regra de preparação e repita a análise.

In [20]:
obrigatorios = {'id', 'texto', 'rotulo'}
problemas_schema = []
problemas_schema.extend(f'Campo ausente: {campo}' for campo in obrigatorios - set(df_original.columns))
if not problemas_schema:
    if df_original[list(obrigatorios)].isna().any().any():
        problemas_schema.append('Existe valor vazio em id, texto ou rotulo')
    if df_original['id'].astype(str).duplicated().any():
        problemas_schema.append('Existem ids duplicados')
    rotulos_encontrados = set(df_original['rotulo'].astype(str).str.strip().str.casefold())
    invalidos = rotulos_encontrados - ROTULOS
    if invalidos:
        problemas_schema.append(f'Rótulos inválidos: {sorted(invalidos)}')

status_schema = 'PASS' if not problemas_schema else 'BLOCKED'
print(status_schema)
display(pd.DataFrame({'problema': problemas_schema or ['Nenhum problema encontrado']}))
salvar_json_sem_sobrescrever_na_pasta_relatorios('02_schema.json', {'passo': 2, 'status': status_schema, 'problemas': problemas_schema})
assert not problemas_schema, 'O schema está bloqueado. Corrija os dados antes de continuar.'

PASS


,problema
0,Nenhum problema encontrado


## Passo 3 — Organizar instrução, contexto e resposta

Agora deixamos explícito o que queremos que o modelo aprenda:

- `instruction`: o que o modelo deve fazer;
- `context`: o texto que será analisado;
- `response`: a resposta esperada.

In [21]:
INSTRUCAO = 'Classifique o sentimento predominante do texto como negativo, neutro ou positivo.'

organizados = pd.DataFrame({
    'id': df_original['id'].astype(str).str.strip(),
    'instruction': INSTRUCAO,
    'context': df_original['texto'].astype(str),
    'response': df_original['rotulo'].astype(str).str.strip().str.casefold(),
})
display(organizados.head())
salvar_jsonl_sem_sobrescrever(TRABALHO / '03_organizados.jsonl', organizados.to_dict(orient='records'))

,id,instruction,context,response
0,1,Classifique o sentimento predominante do texto...,Acordei animado e com vontade de começar o dia.,positivo
1,2,Classifique o sentimento predominante do texto...,Foi muito bom conversar com meus amigos ontem.,positivo
2,3,Classifique o sentimento predominante do texto...,Estou orgulhoso do resultado que consegui no t...,positivo
3,4,Classifique o sentimento predominante do texto...,Hoje encontrei uma solução para um problema di...,positivo
4,5,Classifique o sentimento predominante do texto...,Sinto esperança de que as coisas vão melhorar.,positivo


WindowsPath('c:/Users/mdbaa/development/alura/alura-llama-factory/EscutIA/dataset/dados/trabalho/03_organizados.jsonl')

## Passo 4 — Limpar e normalizar os dados

Vamos corrigir somente formatação: espaços extras, Unicode e capitalização dos rótulos. Registros que não podem ser usados são preservados em uma lista de rejeitados com o motivo. Não vamos inventar conteúdo.

In [22]:
limpos, rejeitados_04 = [], []
for registro in organizados.to_dict(orient='records'):
    novo = {**registro}
    novo['instruction'] = normalizar_texto(novo['instruction'])
    novo['context'] = normalizar_texto(novo['context'])
    novo['response'] = normalizar_texto(novo['response']).casefold()
    motivos = []
    if not novo['id']: motivos.append('id_vazio')
    if not novo['context']: motivos.append('contexto_vazio')
    if novo['response'] not in ROTULOS: motivos.append('rotulo_invalido')
    if motivos: rejeitados_04.append({'registro': novo, 'motivos': motivos})
    else: limpos.append(novo)

print(f'Mantidos: {len(limpos)} | Rejeitados: {len(rejeitados_04)}')
salvar_jsonl_sem_sobrescrever(TRABALHO / '04_limpos.jsonl', limpos)
salvar_jsonl_sem_sobrescrever(TRABALHO / '04_rejeitados.jsonl', rejeitados_04)
salvar_json_sem_sobrescrever_na_pasta_relatorios('04_limpeza.json', {'passo': 4, 'mantidos': len(limpos), 'rejeitados': len(rejeitados_04)})

Mantidos: 3063 | Rejeitados: 0


WindowsPath('c:/Users/mdbaa/development/alura/alura-llama-factory/EscutIA/dataset/dados/relatorios/04_limpeza.json')

## Passo 5 — Remover duplicidades e exemplos inconsistentes

Textos repetidos podem fazer o modelo decorar exemplos. Textos iguais com rótulos diferentes são conflitos que precisam de decisão humana. Vamos separar esses casos sem apagar a evidência.

In [23]:
grupos = {}
for registro in limpos:
    grupos.setdefault(chave_comparacao(registro['context']), []).append(registro)

sem_duplicatas, rejeitados_05 = [], []
for grupo in grupos.values():
    rotulos_do_grupo = {registro['response'] for registro in grupo}
    if len(rotulos_do_grupo) > 1:
        rejeitados_05.extend({'registro': registro, 'motivo': 'CONFLITO_DE_ROTULO'} for registro in grupo)
    else:
        sem_duplicatas.append(grupo[0])
        rejeitados_05.extend({'registro': registro, 'motivo': 'DUPLICATA_DE_TEXTO'} for registro in grupo[1:])

print(f'Mantidos: {len(sem_duplicatas)} | Separados para revisão: {len(rejeitados_05)}')
salvar_jsonl_sem_sobrescrever(TRABALHO / '05_sem_duplicatas.jsonl', sem_duplicatas)
salvar_jsonl_sem_sobrescrever(TRABALHO / '05_rejeitados.jsonl', rejeitados_05)
salvar_json_sem_sobrescrever_na_pasta_relatorios('05_duplicidades.json', {'passo': 5, 'mantidos': len(sem_duplicatas), 'rejeitados': len(rejeitados_05)})

Mantidos: 3057 | Separados para revisão: 6


WindowsPath('c:/Users/mdbaa/development/alura/alura-llama-factory/EscutIA/dataset/dados/relatorios/05_duplicidades.json')

## Passo 6 — Verificar dados sensíveis, incorretos ou fora do domínio

Este detector é apenas uma triagem didática. Ele sinaliza e-mails, telefones, CPF e termos relacionados a crises, violência ou abuso. No domínio da EscutIA, um texto sensível pode ser relevante, então não será excluído automaticamente.

In [24]:
PADROES = {
    'email': re.compile(r'\b[^\s@]+@[^\s@]+\.[^\s@]+\b'),
    'telefone': re.compile(r'(?:\+?55\s*)?(?:\(?\d{2}\)?\s*)?9?\d{4}[-\s]?\d{4}'),
    'cpf': re.compile(r'\b\d{3}[.\s]?\d{3}[.\s]?\d{3}[-\s]?\d{2}\b'),
    'termos_de_crise': re.compile(r'\b(suic[ií]d|autoagress[aã]o|me matar|viol[eê]ncia|abuso)\w*\b', re.IGNORECASE),
}

analisados = []
for registro in sem_duplicatas:
    texto = f"{registro['context']} {registro['response']}"
    alertas = [nome for nome, padrao in PADROES.items() if padrao.search(texto)]
    analisados.append({**registro, 'alertas': alertas, 'status_revisao': 'REVISAR' if alertas else 'OK'})

df_alertas = pd.DataFrame([r for r in analisados if r['status_revisao'] == 'REVISAR'])
print(f'Registros com alerta: {len(df_alertas)}')
display(df_alertas if not df_alertas.empty else pd.DataFrame({'resultado': ['Nenhum alerta encontrado no exemplo']}))
salvar_jsonl_sem_sobrescrever(TRABALHO / '06_conteudo_analisado.jsonl', analisados)
salvar_json_sem_sobrescrever_na_pasta_relatorios('06_conteudo.json', {'passo': 6, 'registros_com_alerta': len(df_alertas), 'alertas': df_alertas['alertas'].tolist() if not df_alertas.empty else []})

Registros com alerta: 7


,id,instruction,context,response,alertas,status_revisao
0,209,Classifique o sentimento predominante do texto...,Por onde apresentamos a peça temos recebido vá...,neutro,[termos_de_crise],REVISAR
1,578,Classifique o sentimento predominante do texto...,"O desarmamento não acaba com a violência, assi...",neutro,[termos_de_crise],REVISAR
2,1214,Classifique o sentimento predominante do texto...,"Sexo, sangue, violência. Não é Game of Thrones...",neutro,[termos_de_crise],REVISAR
3,2429,Classifique o sentimento predominante do texto...,Querem evitar ser vítima de abuso. Mas e cm se...,neutro,[termos_de_crise],REVISAR
4,2600,Classifique o sentimento predominante do texto...,Já contei una 83392991 cortes de cabelo difere...,neutro,[telefone],REVISAR
5,2714,Classifique o sentimento predominante do texto...,#Encontro Na maioria das vezes a violencia est...,neutro,[termos_de_crise],REVISAR
6,2732,Classifique o sentimento predominante do texto...,Duplicou a violência no BR graças ao desarmame...,neutro,[termos_de_crise],REVISAR


WindowsPath('c:/Users/mdbaa/development/alura/alura-llama-factory/EscutIA/dataset/dados/relatorios/06_conteudo.json')

### Checkpoint humano antes da divisão

Agora o notebook fará a revisão dentro da própria aula. Para cada registro com alerta, confira o texto e escolha uma ação:

- `M` — manter o registro;
- `R` — remover somente da cópia derivada;
- `T` — corrigir o texto na cópia derivada;
- `L` — corrigir o rótulo na cópia derivada.

O arquivo `dados/dataset.csv` nunca é alterado.

In [25]:
base_para_divisao = []
decisoes_revisao = []

for registro in analisados:
    if registro['status_revisao'] == 'OK':
        base_para_divisao.append(registro)
        continue

    print('\n' + '=' * 80)
    print(f"ID: {registro['id']}")
    print(f"Texto: {registro['context']}")
    print(f"Rótulo atual: {registro['response']}")
    print(f"Alertas: {', '.join(registro['alertas'])}")
    while True:
        decisao = input('Escolha M=manter, R=remover, T=corrigir texto ou L=corrigir rótulo: ').strip().upper()
        if decisao in {'M', 'R', 'T', 'L'}:
            break
        print('Opção inválida. Digite M, R, T ou L.')

    novo = {**registro}
    justificativa = input('Justificativa curta: ').strip()
    if decisao == 'R':
        novo['status_revisao'] = 'REMOVIDO_MANUALMENTE'
    elif decisao == 'T':
        novo['context'] = input('Digite o texto corrigido: ').strip()
        novo['status_revisao'] = 'CORRIGIDO_MANUALMENTE'
    elif decisao == 'L':
        while True:
            novo_rotulo = input('Digite o novo rótulo (negativo/neutro/positivo): ').strip().casefold()
            if novo_rotulo in ROTULOS:
                break
            print('Rótulo inválido.')
        novo['response'] = novo_rotulo
        novo['status_revisao'] = 'CORRIGIDO_MANUALMENTE'
    else:
        novo['status_revisao'] = 'APROVADO_MANUALMENTE'

    novo['decisao_revisao'] = decisao
    novo['justificativa_revisao'] = justificativa
    decisao_registro = {'id': novo['id'], 'decisao': decisao, 'justificativa': justificativa}
    if decisao == 'T':
        decisao_registro['replacement_text'] = novo['context']
    if decisao == 'L':
        decisao_registro['replacement_label'] = novo['response']
    decisoes_revisao.append(decisao_registro)
    if novo['status_revisao'] != 'REMOVIDO_MANUALMENTE':
        base_para_divisao.append(novo)

display(pd.DataFrame(decisoes_revisao) if decisoes_revisao else pd.DataFrame({'resultado': ['Nenhum alerta precisou de revisão manual.']}))
salvar_jsonl_sem_sobrescrever(TRABALHO / '06_revisados.jsonl', base_para_divisao)
salvar_json_sem_sobrescrever_na_pasta_relatorios('06_revisao_manual.json', {'passo': 6, 'decisoes': decisoes_revisao, 'mantidos_para_divisao': len(base_para_divisao)})
print(f'Registros liberados para a divisão: {len(base_para_divisao)}')


ID: 209
Texto: Por onde apresentamos a peça temos recebido vários relatos de abusos, pessoas se abrindo pela primeira vez #Encontro #HospitalDeBonecas
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 578
Texto: O desarmamento não acaba com a violência, assim como a proibição da maconha não acaba com o tráfico. Me julguem #TheNoite
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 1214
Texto: Sexo, sangue, violência. Não é Game of Thrones, é #MasterChefBR
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 2429
Texto: Querem evitar ser vítima de abuso. Mas e cm se evita q novos abusadores surjam? Ser um abusador tem uma causa. Temos q trata-la!! #Encontro
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 2600
Texto: Já contei una 83392991 cortes de cabelo diferente da Fátima 😂 #VideoShowAoVivo
Rótulo atual: neutro
Alertas: telefone

ID: 2714
Texto: #Encontro Na maioria das vezes a violencia esta camuflada dentro de casa
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 2732
Texto: 

,id,decisao,justificativa
0,209,M,
1,578,M,
2,1214,M,
3,2429,M,
4,2600,M,
5,2714,M,
6,2732,M,


Registros liberados para a divisão: 3057


## Passo 7 — Separar treinamento, validação e avaliação

A divisão será estratificada para preservar a proporção dos três sentimentos. A avaliação ficará separada para ser usada somente na comparação final com o modelo base.

In [26]:
treino_validacao, avaliacao = train_test_split(
    base_para_divisao, test_size=0.20, random_state=42,
    stratify=[r['response'] for r in base_para_divisao],
)
treino, validacao = train_test_split(
    treino_validacao, test_size=0.25, random_state=42,
    stratify=[r['response'] for r in treino_validacao],
)

conjuntos = {'treino': treino, 'validacao': validacao, 'avaliacao': avaliacao}
resumo_divisao = pd.DataFrame({
    nome: pd.Series(Counter(registros[i]['response'] for i in range(len(registros))))
    for nome, registros in conjuntos.items()
}).fillna(0).astype(int)
display(resumo_divisao)
for nome, registros in conjuntos.items():
    salvar_jsonl_sem_sobrescrever(TRABALHO / f'07_{nome}.jsonl', registros)
salvar_json_sem_sobrescrever_na_pasta_relatorios('07_divisao.json', {'passo': 7, 'seed': 42, 'quantidades': {nome: len(registros) for nome, registros in conjuntos.items()}, 'rotulos': resumo_divisao.to_dict()})

,treino,validacao,avaliacao
negativo,612,204,204
neutro,610,204,204
positivo,611,204,204


WindowsPath('c:/Users/mdbaa/development/alura/alura-llama-factory/EscutIA/dataset/dados/relatorios/07_divisao.json')

## Passo 8 — Verificar vazamento entre os conjuntos

Nenhum texto normalizado pode aparecer em mais de um conjunto. Se houver sobreposição, o resultado da avaliação poderá parecer melhor do que realmente é.

In [27]:
chaves_conjuntos = {
    nome: {chave_comparacao(registro['context']) for registro in registros}
    for nome, registros in conjuntos.items()
}
sobreposicoes = {}
for esquerda, direita in [('treino', 'validacao'), ('treino', 'avaliacao'), ('validacao', 'avaliacao')]:
    comuns = chaves_conjuntos[esquerda] & chaves_conjuntos[direita]
    sobreposicoes[f'{esquerda}_x_{direita}'] = len(comuns)

display(pd.Series(sobreposicoes, name='textos_sobrepostos').to_frame())
status_vazamento = 'PASS' if not any(sobreposicoes.values()) else 'BLOCKED'
salvar_json_sem_sobrescrever_na_pasta_relatorios('08_vazamento.json', {'passo': 8, 'status': status_vazamento, 'sobreposicoes': sobreposicoes})
assert status_vazamento == 'PASS', 'Existe vazamento entre os conjuntos. Corrija a divisão antes de continuar.'

,textos_sobrepostos
treino_x_validacao,0
treino_x_avaliacao,0
validacao_x_avaliacao,0


## Passo 9 — Congelar o conjunto de avaliação

A avaliação será copiada para `dados/preparados` e receberá um hash. Depois deste ponto, ela não deve ser alterada durante os ajustes do treinamento.

In [28]:
avaliacao_congelada = PREPARADOS / 'avaliacao_congelada.jsonl'
salvar_jsonl_sem_sobrescrever(avaliacao_congelada, avaliacao)
manifesto_avaliacao = {
    'passo': 9,
    'arquivo': str(avaliacao_congelada),
    'sha256': sha256_arquivo(avaliacao_congelada),
    'registros': len(avaliacao),
    'congelado_em_utc': datetime.now(timezone.utc).isoformat(),
}
print(f"SHA-256: {manifesto_avaliacao['sha256']}")
salvar_json_sem_sobrescrever_na_pasta_relatorios('09_avaliacao_congelada.json', manifesto_avaliacao)

SHA-256: 948a8c4568b0b0eefd97f532a071c1d7af24aa9650fbebc6ba77ac351013e801


WindowsPath('c:/Users/mdbaa/development/alura/alura-llama-factory/EscutIA/dataset/dados/relatorios/09_avaliacao_congelada.json')

## Passo 10 — Converter para os formatos do LLaMA-Factory

O formato final desta aula terá duas representações equivalentes: Alpaca, com `instruction`, `input` e `output`, e conversacional ShareGPT/OpenAI, com `messages` e os papéis `system`, `user` e `assistant`. O `dataset_info.json` registra ambos para o LLaMA-Factory. A turma deve escolher uma representação por experimento, sem misturá-las.

In [29]:
def converter_para_alpaca(registros):
    return [
        {'instruction': registro['instruction'], 'input': registro['context'], 'output': registro['response']}
        for registro in registros
    ]

def converter_para_conversacional(registros):
    return [
        {'messages': [
            {'role': 'system', 'content': 'Você classifica sentimentos em textos em português.'},
            {'role': 'user', 'content': registro['instruction'] + '\n\nTexto: ' + registro['context']},
            {'role': 'assistant', 'content': registro['response']},
        ]}
        for registro in registros
    ]

nomes_finais = {'treino': 'escutia_train.json', 'validacao': 'escutia_validation.json', 'avaliacao': 'escutia_evaluation.json'}
nomes_conversacionais = {'treino': 'escutia_train_conversacional.json', 'validacao': 'escutia_validation_conversacional.json', 'avaliacao': 'escutia_evaluation_conversacional.json'}
arquivos_finais = {}
arquivos_conversacionais = {}
for nome, registros in conjuntos.items():
    arquivo = PREPARADOS / nomes_finais[nome]
    arquivo.write_text(json.dumps(converter_para_alpaca(registros), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    arquivos_finais[nome] = arquivo
    arquivo_conversacional = PREPARADOS / nomes_conversacionais[nome]
    arquivo_conversacional.write_text(json.dumps(converter_para_conversacional(registros), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    arquivos_conversacionais[nome] = arquivo_conversacional

dataset_info = {}
for nome in nomes_finais:
    dataset_info[f'escutia_{nome}'] = {
        'file_name': nomes_finais[nome],
        'columns': {'prompt': 'instruction', 'query': 'input', 'response': 'output'},
    }
    dataset_info[f'escutia_{nome}_conversacional'] = {
        'file_name': nomes_conversacionais[nome],
        'formatting': 'sharegpt',
        'columns': {'messages': 'messages'},
    }
info_path = PREPARADOS / 'dataset_info.json'
info_path.write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame(converter_para_alpaca(treino)).head())
display(pd.DataFrame(converter_para_conversacional(treino)).head())

,instruction,input,output
0,Classifique o sentimento predominante do texto...,"O Padre Fabio de Melo me passa uma paz, deve s...",positivo
1,Classifique o sentimento predominante do texto...,"#VideoShowAoVivo Marinho da Bahia ligado , pro...",positivo
2,Classifique o sentimento predominante do texto...,E eu que chorei com Maiara e Maraisa cantando ...,positivo
3,Classifique o sentimento predominante do texto...,Irmãs Galvão que amorzinho! Minha avó gostava ...,positivo
4,Classifique o sentimento predominante do texto...,"O meu Douglas saiu, eu não estou acreditando #...",negativo


,messages
0,"[{'role': 'system', 'content': 'Você classific..."
1,"[{'role': 'system', 'content': 'Você classific..."
2,"[{'role': 'system', 'content': 'Você classific..."
3,"[{'role': 'system', 'content': 'Você classific..."
4,"[{'role': 'system', 'content': 'Você classific..."


## Passo 11 — Validar o dataset final

A etapa final verifica schema, rótulos, arquivos, isolamento entre conjuntos, registro `dataset_info.json` e existência da avaliação congelada. A preparação também gera linhagem, rejeitados, manifesto de origem e uma variante conversacional ShareGPT/OpenAI em `dados/preparados`.

A decisão `PRONTO_PARA_TREINAMENTO` deste notebook é substituída pelo gate `DATA_READY_FOR_SFT` em `dados/relatorios/11_validacao_final.json`. Esse gate significa somente que a Parte de Preparação de Dados foi concluída; ainda será necessário revisar modelo, template, hardware, estratégia LoRA/QLoRA e autorização do treinamento.

In [30]:
schema_final = {
    'type': 'object',
    'required': ['instruction', 'input', 'output'],
    'additionalProperties': False,
    'properties': {
        'instruction': {'type': 'string', 'minLength': 1},
        'input': {'type': 'string', 'minLength': 1},
        'output': {'type': 'string', 'enum': sorted(ROTULOS)},
    },
}
validador = Draft202012Validator(schema_final)
def validar_conversa(registro):
    mensagens = registro.get('messages', [])
    papeis = [mensagem.get('role') for mensagem in mensagens]
    conteudos = [mensagem.get('content') for mensagem in mensagens]
    return papeis == ['system', 'user', 'assistant'] and all(isinstance(conteudo, str) and conteudo.strip() for conteudo in conteudos) and mensagens[-1]['content'] in ROTULOS
resultados_finais = []
dados_finais = {}
for nome, arquivo in arquivos_finais.items():
    registros = json.loads(arquivo.read_text(encoding='utf-8'))
    dados_finais[nome] = registros
    erros = [erro.message for registro in registros for erro in validador.iter_errors(registro)]
    resultados_finais.append({'checagem': f'schema_{nome}', 'status': 'PASS' if registros and not erros else 'FAIL', 'registros': len(registros), 'erros': erros[:10]})

for nome, arquivo in arquivos_conversacionais.items():
    registros = json.loads(arquivo.read_text(encoding='utf-8'))
    erros = [registro for registro in registros if not validar_conversa(registro)]
    resultados_finais.append({'checagem': f'conversacional_{nome}', 'status': 'PASS' if registros and not erros else 'FAIL', 'registros': len(registros), 'erros': len(erros)})

chaves_finais = {nome: {chave_comparacao(registro['input']) for registro in registros} for nome, registros in dados_finais.items()}
for esquerda, direita in [('treino', 'validacao'), ('treino', 'avaliacao'), ('validacao', 'avaliacao')]:
    comuns = chaves_finais[esquerda] & chaves_finais[direita]
    resultados_finais.append({'checagem': f'isolamento_{esquerda}_x_{direita}', 'status': 'PASS' if not comuns else 'FAIL', 'sobreposicoes': len(comuns)})

info = json.loads((PREPARADOS / 'dataset_info.json').read_text(encoding='utf-8'))
info_ok = all(info[f'escutia_{nome}']['file_name'] == nomes_finais[nome] and info[f'escutia_{nome}_conversacional']['file_name'] == nomes_conversacionais[nome] for nome in nomes_finais)
resultados_finais.append({'checagem': 'dataset_info', 'status': 'PASS' if info_ok else 'FAIL'})
resultados_finais.append({'checagem': 'avaliacao_congelada', 'status': 'PASS' if (PREPARADOS / 'avaliacao_congelada.jsonl').exists() else 'FAIL'})

pronto = all(resultado['status'] == 'PASS' for resultado in resultados_finais)
decisao = 'DATA_READY_FOR_SFT' if pronto else 'BLOQUEADO'
display(pd.DataFrame(resultados_finais))
relatorio_final = salvar_json_sem_sobrescrever_na_pasta_relatorios('11_validacao_final.json', {'passo': 11, 'decisao': decisao, 'resultados': resultados_finais, 'distribuicao': {nome: dict(Counter(registro['output'] for registro in registros)) for nome, registros in dados_finais.items()}})
print(f'DECISÃO: {decisao}')
print(f'ARQUIVO GERADO: {relatorio_final}')
display(json.loads(relatorio_final.read_text(encoding='utf-8')))
assert pronto, 'O dataset está bloqueado. Corrija os itens FAIL antes de avançar.'

,checagem,status,registros,erros,sobreposicoes
0,schema_treino,PASS,1833.0,[],NaN
1,schema_validacao,PASS,612.0,[],NaN
2,schema_avaliacao,PASS,612.0,[],NaN
3,conversacional_treino,PASS,1833.0,0,NaN
4,conversacional_validacao,PASS,612.0,0,NaN
5,conversacional_avaliacao,PASS,612.0,0,NaN
6,isolamento_treino_x_validacao,PASS,NaN,NaN,0.0
7,isolamento_treino_x_avaliacao,PASS,NaN,NaN,0.0
8,isolamento_validacao_x_avaliacao,PASS,NaN,NaN,0.0
9,dataset_info,PASS,NaN,NaN,NaN


DECISÃO: DATA_READY_FOR_SFT
ARQUIVO GERADO: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\dataset\dados\relatorios\11_validacao_final.json


{'passo': 11,
 'decisao': 'DATA_READY_FOR_SFT',
 'resultados': [{'checagem': 'schema_treino',
   'status': 'PASS',
   'registros': 1833,
   'erros': []},
  {'checagem': 'schema_validacao',
   'status': 'PASS',
   'registros': 612,
   'erros': []},
  {'checagem': 'schema_avaliacao',
   'status': 'PASS',
   'registros': 612,
   'erros': []},
  {'checagem': 'conversacional_treino',
   'status': 'PASS',
   'registros': 1833,
   'erros': 0},
  {'checagem': 'conversacional_validacao',
   'status': 'PASS',
   'registros': 612,
   'erros': 0},
  {'checagem': 'conversacional_avaliacao',
   'status': 'PASS',
   'registros': 612,
   'erros': 0},
  {'checagem': 'isolamento_treino_x_validacao',
   'status': 'PASS',
   'sobreposicoes': 0},
  {'checagem': 'isolamento_treino_x_avaliacao',
   'status': 'PASS',
   'sobreposicoes': 0},
  {'checagem': 'isolamento_validacao_x_avaliacao',
   'status': 'PASS',
   'sobreposicoes': 0},
  {'checagem': 'dataset_info', 'status': 'PASS'},
  {'checagem': 'avaliacao

## Checklist da Parte 2

- [ ] Dataset original preservado e origem/licença registradas.
- [ ] Schema original e schema final validados.
- [ ] Instrução, contexto e resposta organizados.
- [ ] Texto limpo e normalizado sem inventar conteúdo.
- [ ] Duplicidades, conflitos e quase duplicatas revisados.
- [ ] PII, idioma e conteúdo sensível analisados por uma pessoa.
- [ ] Decisões humanas justificadas e linhagem preservada.
- [ ] Treino, validação e avaliação separados de forma estratificada.
- [ ] Vazamento entre conjuntos inexistente.
- [ ] Avaliação congelada com hash.
- [ ] Formatos Alpaca e conversacional gerados.
- [ ] `dataset_info.json` registrado para o LLaMA-Factory.
- [ ] Resultado final: `DATA_READY_FOR_SFT`.

Quando todos os itens estiverem confirmados, a Parte 2 terminou. Execute as células de 1 a 11 na ordem; não há uma célula única que substitua as etapas. O próximo módulo só poderá tratar da estratégia de LoRA depois que `dados/relatorios/11_validacao_final.json` estiver em `DATA_READY_FOR_SFT`.